In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [19]:
# Load the Diabetes dataset
diabetes = load_diabetes()
X = diabetes.data
Y = diabetes.target

import pandas as pd

feature_names = diabetes.feature_names
df_X = pd.DataFrame(X, columns=feature_names)
df_Y = pd.DataFrame(Y, columns=['target'])
df_diabetes = pd.concat([df_X, df_Y], axis=1)

print(f"Total data: {len(X)}")

print("\nFirst 5 rows of the Diabetes dataset:")
print(df_diabetes.head())

Total data: 442

First 5 rows of the Diabetes dataset:
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  


# (a) Train/Validation Split (75% / 25%)

In [20]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.25, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples\n")

Training set size: 331 samples
Validation set size: 111 samples



# (b) Least Squares From Scratch

In [21]:
def least_squares_fit(X_in, Y_in):
    # Add a column of ones for the bias term
    X_b = np.c_[np.ones((X_in.shape[0], 1)), X_in]
    # Apply the normal equation: beta = (X^T * X)^-1 * X^T * Y
    beta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(Y_in)
    return beta

# Get coefficients using training set
beta_scratch = least_squares_fit(X_train, Y_train)

# Predict on validation set
X_val_b = np.c_[np.ones((X_val.shape[0], 1)), X_val]
Y_pred_scratch = X_val_b.dot(beta_scratch)

# Calculate metrics manually
mse_scratch = mean_squared_error(Y_val, Y_pred_scratch)
r2_scratch = r2_score(Y_val, Y_pred_scratch)

print(f"MSE: {mse_scratch:.4f}")
print(f"R2 Score: {r2_scratch:.4f}\n")

# # Plotting Y_True vs Y_Pred
# plt.figure(figsize=(8, 5))
# plt.scatter(Y_val, Y_pred_scratch, alpha=0.7, color='blue', edgecolors='k')
# plt.plot([Y_val.min(), Y_val.max()], [Y_val.min(), Y_val.max()], 'r--', lw=2)
# plt.title("Validation Dataset: True vs Predicted (From Scratch)")
# plt.xlabel("True Values")
# plt.ylabel("Predicted Values")
# plt.grid(True)
# plt.show()

MSE: 2848.3107
R2 Score: 0.4849



# (c) Least Squares using scikit-learn

In [22]:
lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)
Y_pred_sklearn = lr_model.predict(X_val)

mse_sklearn = mean_squared_error(Y_val, Y_pred_sklearn)
r2_sklearn = r2_score(Y_val, Y_pred_sklearn)

print(f"MSE: {mse_sklearn:.4f}")
print(f"R2 Score: {r2_sklearn:.4f}")

MSE: 2848.3107
R2 Score: 0.4849


# (d) Forward Stepwise Selection

In [23]:
def forward_stepwise(X_tr, Y_tr, X_v, Y_v):
    features = list(range(X_tr.shape[1]))
    selected_features = []
    best_overall_mse = float('inf')

    while features:
        best_step_mse = float('inf')
        best_feature_to_add = None

        for feature in features:
            candidate_features = selected_features + [feature]
            model = LinearRegression().fit(X_tr[:, candidate_features], Y_tr)
            preds = model.predict(X_v[:, candidate_features])
            mse = mean_squared_error(Y_v, preds)

            if mse < best_step_mse:
                best_step_mse = mse
                best_feature_to_add = feature

        # If adding the best feature in this step improves overall MSE, keep it
        if best_step_mse < best_overall_mse:
            best_overall_mse = best_step_mse
            selected_features.append(best_feature_to_add)
            features.remove(best_feature_to_add)
        else:
            break # No further improvement

    # Final model evaluation with best features
    model = LinearRegression().fit(X_tr[:, selected_features], Y_tr)
    final_preds = model.predict(X_v[:, selected_features])
    final_r2 = r2_score(Y_v, final_preds)

    return selected_features, best_overall_mse, final_r2

fwd_features, fwd_mse, fwd_r2 = forward_stepwise(X_train, Y_train, X_val, Y_val)
print(f"Best Feature Indices: {fwd_features}")
print(f"MSE: {fwd_mse:.4f}")
print(f"R2 Score: {fwd_r2:.4f}\n")

Best Feature Indices: [8, 2, 1, 9, 6]
MSE: 2645.2735
R2 Score: 0.5216



# (e) Backward Stepwise Selection

In [24]:
def backward_stepwise(X_tr, Y_tr, X_v, Y_v):
    selected_features = list(range(X_tr.shape[1]))

    # Baseline with all features
    model = LinearRegression().fit(X_tr, Y_tr)
    best_overall_mse = mean_squared_error(Y_v, model.predict(X_v))

    while len(selected_features) > 1:
        best_step_mse = float('inf')
        feature_to_remove = None

        for feature in selected_features:
            candidate_features = selected_features.copy()
            candidate_features.remove(feature)

            model = LinearRegression().fit(X_tr[:, candidate_features], Y_tr)
            preds = model.predict(X_v[:, candidate_features])
            mse = mean_squared_error(Y_v, preds)

            if mse < best_step_mse:
                best_step_mse = mse
                feature_to_remove = feature

        if best_step_mse < best_overall_mse:
            best_overall_mse = best_step_mse
            selected_features.remove(feature_to_remove)
        else:
            break # Removing more features doesn't help

    # Final model evaluation
    model = LinearRegression().fit(X_tr[:, selected_features], Y_tr)
    final_preds = model.predict(X_v[:, selected_features])
    final_r2 = r2_score(Y_v, final_preds)

    return selected_features, best_overall_mse, final_r2

bwd_features, bwd_mse, bwd_r2 = backward_stepwise(X_train, Y_train, X_val, Y_val)
print(f"Best Feature Indices: {bwd_features}")
print(f"MSE: {bwd_mse:.4f}")
print(f"R2 Score: {bwd_r2:.4f}\n")

Best Feature Indices: [1, 2, 4, 5, 8, 9]
MSE: 2659.3848
R2 Score: 0.5191



# (f) Ridge Regression

In [25]:
lambdas = [0.01, 0.1, 1, 10, 100]

best_ridge_mse = float('inf')
best_ridge_lam = None

for lam in lambdas:
    ridge = Ridge(alpha=lam)
    ridge.fit(X_train, Y_train)
    preds = ridge.predict(X_val)
    mse = mean_squared_error(Y_val, preds)
    r2 = r2_score(Y_val, preds)
    print(f"Lambda {lam:5}: MSE = {mse:.4f}, R2 = {r2:.4f}")

    if mse < best_ridge_mse:
        best_ridge_mse = mse
        best_ridge_lam = lam

print(f"--> Best Lambda for Ridge: {best_ridge_lam}\n")

Lambda  0.01: MSE = 2836.4072, R2 = 0.4871
Lambda   0.1: MSE = 2810.0386, R2 = 0.4918
Lambda     1: MSE = 3105.4721, R2 = 0.4384
Lambda    10: MSE = 4664.7173, R2 = 0.1564
Lambda   100: MSE = 5479.4526, R2 = 0.0091
--> Best Lambda for Ridge: 0.1



# (g) Lasso Regression

In [26]:
best_lasso_mse = float('inf')
best_lasso_lam = None

for lam in lambdas:
    lasso = Lasso(alpha=lam)
    lasso.fit(X_train, Y_train)
    preds = lasso.predict(X_val)
    mse = mean_squared_error(Y_val, preds)
    r2 = r2_score(Y_val, preds)
    print(f"Lambda {lam:5}: MSE = {mse:.4f}, R2 = {r2:.4f}")

    if mse < best_lasso_mse:
        best_lasso_mse = mse
        best_lasso_lam = lam

print(f"--> Best Lambda for Lasso: {best_lasso_lam}")

Lambda  0.01: MSE = 2831.0002, R2 = 0.4880
Lambda   0.1: MSE = 2753.9218, R2 = 0.5020
Lambda     1: MSE = 3433.1555, R2 = 0.3791
Lambda    10: MSE = 5607.1979, R2 = -0.0140
Lambda   100: MSE = 5607.1979, R2 = -0.0140
--> Best Lambda for Lasso: 0.1
